# Chat Completions — gọi OpenAI API

Notebook này cài đặt hàm `call_chat_completion(...)` để gọi Chat Completions, đo latency bằng `time.perf_counter()` và trả `(response_text, latency_seconds)`.

API key được đọc từ biến môi trường, tuyệt đối không ghi key vào notebook:

```powershell
$env:OPENAI_API_KEY = 'sk-...'
$env:OPENAI_MODEL = 'gpt-6.1-terra'       # tùy chọn
$env:OPENAI_BASE_URL = 'https://api.example.com/v1'  # tùy chọn khi gateway cấp URL riêng
```


In [ ]:
# Chạy ô này một lần nếu môi trường chưa có SDK.
%pip install "openai>=1.66,<4

## Mã triển khai

- `provider_kwargs(max_tokens)` gửi `max_completion_tokens` thay vì `max_tokens` và đặt `reasoning_effort="none"`.
- GPT-6 chỉ cho phép `temperature/top_p` khi reasoning effort là `none`.
- Latency được đo bằng perf counter và chỉ bao quanh API call.
- Nếu provider trả `message.content = None`, hàm trả chuỗi rỗng thay vì `None`.


In [ ]:
import os
import time
from typing import Any

try:
    from openai import OpenAI
except ImportError as exc:
    raise ImportError("Chưa có SDK openai. Chạy ô cài đặt rồi restart kernel.") from exc

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6.1-terra")


def provider_kwargs(max_tokens: int) -> dict[str, Any]:
    """Trả tham số Chat Completions tương thích GPT-6/OpenAI."""
    if max_tokens <= 0:
        raise ValueError("max_tokens phải là số nguyên dương")
    return {
        "max_completion_tokens": max_tokens,
        "reasoning_effort": "none",
    }


def call_chat_completion(
    prompt: str,
    model: str = OPENAI_MODEL,
    temperature: float = 0.7,
    top_p: float = 0.9,
    max_tokens: int = 256,
) -> tuple[str, float]:
    """Gọi Chat Completions và trả về (response_text, latency_seconds)."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt phải là chuỗi không rỗng")
    if not 0.0 <= temperature <= 2.0:
        raise ValueError("temperature phải nằm trong khoảng 0.0–2.0")
    if not 0.0 <= top_p <= 1.0:
        raise ValueError("top_p phải nằm trong khoảng 0.0–1.0")

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise EnvironmentError("Thiếu OPENAI_API_KEY trong biến môi trường")

    client_kwargs: dict[str, Any] = {"api_key": api_key}
    base_url = os.getenv("OPENAI_BASE_URL")
    if base_url:
        client_kwargs["base_url"] = base_url

    client = OpenAI(**client_kwargs)

    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        **provider_kwargs(max_tokens),
    )
    latency_seconds = time.perf_counter() - start

    response_text = response.choices[0].message.content or ""
    return response_text, latency_seconds


## Ví dụ chạy

Ô dưới chỉ gọi API khi `OPENAI_API_KEY` đã tồn tại. Khi key chưa được cấp, nó bỏ qua cuộc gọi và không phát sinh chi phí.


In [ ]:
TEST_PROMPT = "Hãy giải thích ngắn gọn Chat Completions là gì."

if os.getenv("OPENAI_API_KEY"):
    text, latency = call_chat_completion(TEST_PROMPT)
    print(text)
    print(f"Latency: {latency:.3f} seconds")
else:
    print("Chưa cấu hình OPENAI_API_KEY — bỏ qua cuộc gọi API mẫu.")

## Lưu ý vận hành

- Không lưu API key trong notebook, code, chat history hoặc shell history công khai.
- Khi gateway yêu cầu model khác, truyền `model=...` hoặc đặt `OPENAI_MODEL`.
- Lỗi 400 về `temperature/top_p`: kiểm tra `reasoning_effort="none"`.
- Lỗi 400 về `max_tokens`: dùng đúng `max_completion_tokens` như notebook đã làm.
- Chạy API demo chỉ sau khi key được cấp và biến môi trường đã được nạp vào kernel.
